<a href="https://colab.research.google.com/github/SaadJavedQamar/final-year-project-Vulnbrace-/blob/main/SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from google.colab import files

# Step 1: Upload the JSON dataset file
print("Please upload your JSON dataset file (e.g., SQLiV3_modified.json):")
uploaded = files.upload()  # Upload the file

# Get the uploaded file name
dataset_path = list(uploaded.keys())[0]
print(f"Uploaded file: {dataset_path}")

# Load the JSON file
with open(dataset_path, "r") as file:
    data = json.load(file)

# Convert JSON to Hugging Face Dataset
dataset = Dataset.from_list(data)

# Step 2: Load the tokenizer and model
model_name = "EleutherAI/gpt-neo-125M"  # A smaller model for faster training
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add padding token if it doesn't exist
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Load the model
model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

# Step 3: Tokenize the dataset
def tokenize_function(examples):
    inputs = tokenizer(
        examples["instruction"],  # Adjust to match your JSON keys
        text_pair=examples["output"],  # Adjust to match your JSON keys
        truncation=True,
        padding="max_length",
        max_length=128
    )
    inputs["labels"] = inputs["input_ids"].copy()  # Add labels for causal language modeling
    return inputs

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Reduce dataset size for faster training
small_train_size = 200  # Use a smaller subset for training
small_eval_size = 50    # Use a smaller subset for evaluation
tokenized_dataset = tokenized_dataset.shuffle(seed=42)
train_dataset = tokenized_dataset.select(range(small_train_size))
eval_dataset = tokenized_dataset.select(range(small_train_size, small_train_size + small_eval_size))

# Step 4: Define training arguments
training_args = TrainingArguments(
    output_dir="./results",           # Output directory for model checkpoints
    do_eval=True,                     # Enable evaluation
    learning_rate=5e-5,               # Learning rate
    per_device_train_batch_size=8,    # Larger batch size for faster training
    per_device_eval_batch_size=8,     # Larger batch size for evaluation
    num_train_epochs=1,               # Fewer epochs for faster training
    save_steps=100,                   # Save checkpoint more frequently
    save_total_limit=1,               # Keep only the latest checkpoint
    fp16=True,                        # Mixed precision for faster training (requires GPU)
    logging_dir="./logs",             # Directory for logs
    logging_steps=10,                 # Log every 10 steps
    report_to="none"                  # Disable W&B logging
)

# Step 5: Initialize the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# Step 6: Fine-tune the model
trainer.train()

# Step 7: Save the fine-tuned model
output_dir = "./fine_tuned_gpt_neo_125M"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved to {output_dir}")

# Step 8: Generate SQL queries using the fine-tuned model
def generate_sql(prompt, model, tokenizer, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt").input_ids
    outputs = model.generate(inputs, max_length=max_length, num_return_sequences=1)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example test prompt
test_prompt = "Generate an SQL injection payload to bypass login:"
generated_sql = generate_sql(test_prompt, model, tokenizer)
print(f"Generated SQL Injection Payload:\n{generated_sql}")


Please upload your JSON dataset file (e.g., SQLiV3_modified.json):


Saving SQLiV3_modified.json to SQLiV3_modified.json
Uploaded file: SQLiV3_modified.json


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Map:   0%|          | 0/11293 [00:00<?, ? examples/s]

Step,Training Loss
10,6.200400
20,5.420100


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model saved to ./fine_tuned_gpt_neo_125M
Generated SQL Injection Payload:
Generate an SQL injection payload to bypass login:
                                       


In [4]:
# Step 1: Install necessary libraries (run only once)
!pip install transformers datasets

# Step 2: Upload JSON dataset
from google.colab import files
import json
from datasets import Dataset

uploaded = files.upload()  # Upload SQLiV3_modified.json here
dataset_path = list(uploaded.keys())[0]

# Load JSON data
with open(dataset_path, "r") as f:
    data = json.load(f)

# Convert to Hugging Face Dataset
dataset = Dataset.from_list(data)

# Step 3: Load the tokenizer and model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Step 4: Tokenize the dataset
def tokenize_function(examples):
    inputs = tokenizer(
        examples["instruction"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    targets = tokenizer(
        examples["output"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Step 5: Prepare train and eval splits
tokenized_dataset = tokenized_dataset.shuffle(seed=42)
small_train_size = 200
small_eval_size = 50
train_dataset = tokenized_dataset.select(range(small_train_size))
eval_dataset = tokenized_dataset.select(range(small_train_size, small_train_size + small_eval_size))

# Step 6: Define training arguments
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    do_eval=True,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    save_steps=100,
    save_total_limit=1,
    fp16=False,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none"
)

# Step 7: Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# Step 8: Train the model
trainer.train()

# Step 9: Save model and tokenizer
output_dir = "./fine_tuned_distilbart"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved to {output_dir}")

# Step 10: Generate SQL queries from prompt
import torch

def generate_sql(prompt, model, tokenizer, max_length=50, num_return_sequences=3):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)
    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        temperature=0.7,
        top_k=50,
        top_p=0.9
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

# Example usage
test_prompt = "Generate an SQL injection payload to bypass login:"
print(f"Prompt: {test_prompt}")
generated = generate_sql(test_prompt, model, tokenizer)

print("\nGenerated SQL Injection Payloads:")
for i, g in enumerate(generated, 1):
    print(f"{i}. {g}")


Saving SQLiV3_modified.json to SQLiV3_modified.json


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

Step,Training Loss
10,7.162300
20,2.891900
30,1.528100
40,1.078600
50,0.945200
60,0.823900
70,0.704600


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Model saved to ./fine_tuned_distilbart
Prompt: Generate an SQL injection payload to bypass login:


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/utils.py:1570: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (50). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(



Generated SQL Injection Payloads:
1. 1'  )    (  select '%' or 'r' (  case when   =   0x0x717010101020201010302010301020102
2. 1'  )    (  select '%' or 'r' (  case when   =   0x0x717010101020201010302010301011101
3. 1'  )    (  select '%' or 'r' (  case when   =   0x0x717010101020201010302010301011111
